In [ ]:

#### Data acquisition

In [2]:
import pandas as pd

# Load the Home Credit Default Risk dataset (train.csv)
# Make sure the file is in your working directory or provide full path
df = pd.read_csv("/content/application_train.csv")

# Display the first few rows
display(df.head())

# Print shape and data types
print("Shape of the DataFrame:", df.shape)
print("\nData types of the columns:")
print(df.dtypes)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Shape of the DataFrame: (159213, 122)

Data types of the columns:
SK_ID_CURR                      int64
TARGET                          int64
NAME_CONTRACT_TYPE             object
CODE_GENDER                    object
FLAG_OWN_CAR                   object
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     float64
AMT_REQ_CREDIT_BUREAU_WEEK    float64
AMT_REQ_CREDIT_BUREAU_MON     float64
AMT_REQ_CREDIT_BUREAU_QRT     float64
AMT_REQ_CREDIT_BUREAU_YEAR    float64
Length: 122, dtype: object


In [ ]:
#### Data Pre Processing

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset (replace with your path)
df = pd.read_csv("/content/application_train.csv")

# Separate features (X) and target variable (y)
X = df.drop('TARGET', axis=1)   # In Home Credit dataset, the target column is 'TARGET'
y = df['TARGET']

# Check for missing values
print("Missing values in features:\n", X.isnull().sum())
print("\nMissing values in target:\n", y.isnull().sum())

# Handle missing values (basic strategy: fill with median for numeric, mode for categorical)
X = X.fillna(X.median(numeric_only=True))  # for numeric columns
X = X.fillna(X.mode().iloc[0])             # for categorical columns

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nShape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


Missing values in features:
 SK_ID_CURR                        0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
FLAG_OWN_REALTY                   0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     33097
AMT_REQ_CREDIT_BUREAU_WEEK    33097
AMT_REQ_CREDIT_BUREAU_MON     33097
AMT_REQ_CREDIT_BUREAU_QRT     33097
AMT_REQ_CREDIT_BUREAU_YEAR    33097
Length: 121, dtype: int64

Missing values in target:
 0

Shape of X_train: (195724, 121)
Shape of X_test: (48931, 121)
Shape of y_train: (195724,)
Shape of y_test: (48931,)


In [ ]:
#### Base models training

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Identify categorical and numeric columns
cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(exclude=['object']).columns

# Preprocessor: OneHotEncode categorical, keep numeric as is
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Define base models with preprocessing
log_reg = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

dt_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

knn_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', KNeighborsClassifier())
])

# Train models
log_reg.fit(X_train, y_train)
dt_clf.fit(X_train, y_train)
knn_clf.fit(X_train, y_train)

print("✅ Logistic Regression trained")
print("✅ Decision Tree trained")
print("✅ KNN trained")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ Logistic Regression trained
✅ Decision Tree trained
✅ KNN trained


In [ ]:
### Blending implementation

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ==============================
# 1. Preprocessing
# ==============================
cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# ==============================
# 2. Define Base Models
# ==============================
log_reg = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

dt_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

knn_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', KNeighborsClassifier())
])

# ==============================
# 3. Split Data for Blending
# ==============================
X_train_base, X_blend, y_train_base, y_blend = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42
)

# ==============================
# 4. Train Base Models
# ==============================
log_reg.fit(X_train_base, y_train_base)
dt_clf.fit(X_train_base, y_train_base)
knn_clf.fit(X_train_base, y_train_base)

# ==============================
# 5. Generate Predictions for Blending Set
# ==============================
blend_preds_log_reg = log_reg.predict(X_blend)
blend_preds_dt = dt_clf.predict(X_blend)
blend_preds_knn = knn_clf.predict(X_blend)

# ==============================
# 6. Create Meta Features
# ==============================
X_blend_meta = pd.DataFrame({
    'log_reg_preds': blend_preds_log_reg,
    'dt_preds': blend_preds_dt,
    'knn_preds': blend_preds_knn
})

# ==============================
# 7. Define and Train Meta-Model
# ==============================
meta_model = LogisticRegression(random_state=42, max_iter=1000)
meta_model.fit(X_blend_meta, y_blend)

print("✅ Blending setup and meta-model training complete.")
display(X_blend_meta.head())


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ Blending setup and meta-model training complete.


,log_reg_preds,dt_preds,knn_preds
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0


In [ ]:
####  Stacking implementation

In [18]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ==============================
# 1. Preprocessing
# ==============================
cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# ==============================
# 2. Define Base Models (with preprocessing)
# ==============================
log_reg = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

dt_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

knn_clf = Pipeline([
    ('preprocess', preprocessor),
    ('model', KNeighborsClassifier())
])

# ==============================
# 3. StackingClassifier Setup
# ==============================
estimators = [
    ('lr', log_reg),
    ('dt', dt_clf),
    ('knn', knn_clf)
]

# By default, final_estimator = LogisticRegression()
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42, max_iter=1000)
)

# ==============================
# 4. Train StackingClassifier
# ==============================
stacking_clf.fit(X_train, y_train)

# ==============================
# 5. Predictions on Test Set
# ==============================
final_stacking_predictions = stacking_clf.predict(X_test)

print("✅ Stacking classifier trained and predictions generated.")
print("\nFinal stacking predictions (first 5):", final_stacking_predictions[:5])


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

✅ Stacking classifier trained and predictions generated.

Final stacking predictions (first 5): [0 0 0 0 0]


In [29]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# ==============================
# 1. Separate categorical & numeric columns
# ==============================
categorical_cols = X_train.select_dtypes(include=['object']).columns
numeric_cols = X_train.select_dtypes(exclude=['object']).columns

# ==============================
# 2. Preprocessor (OneHot for categorical, passthrough numeric)
# ==============================
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# ==============================
# 3. AdaBoost with Decision Tree as base estimator
# ==============================
ada_boost_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

# ==============================
# 4. Pipeline (Preprocessing + AdaBoost)
# ==============================
ada_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ada_boost_clf)
])

# ==============================
# 5. Train
# ==============================
ada_pipeline.fit(X_train, y_train)

# ==============================
# 6. Predict
# ==============================
final_adaboost_predictions = ada_pipeline.predict(X_test)

# ==============================
# 7. Evaluate
# ==============================
print("✅ AdaBoost classifier trained and predictions generated.\n")

print("First 5 Predictions:", final_adaboost_predictions[:5])
print("\nAccuracy Score:", accuracy_score(y_test, final_adaboost_predictions))
print("\nClassification Report:\n", classification_report(y_test, final_adaboost_predictions))


✅ AdaBoost classifier trained and predictions generated.

First 5 Predictions: [0 0 0 0 0]

Accuracy Score: 0.9183544174449735

Classification Report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96     44929
           1       0.55      0.01      0.02      4002

    accuracy                           0.92     48931
   macro avg       0.73      0.50      0.49     48931
weighted avg       0.89      0.92      0.88     48931



In [ ]:
### Model evaluation

In [33]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

# Blending (VotingClassifier as proxy)
blend_clf = VotingClassifier(
    estimators=[('lr', log_reg), ('dt', dt_clf), ('knn', knn_clf)],
    voting='soft'
)
blend_clf.fit(X_train, y_train)

# Stacking
stacking_clf = StackingClassifier(
    estimators=[('lr', log_reg), ('dt', dt_clf), ('knn', knn_clf)],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42)
)
stacking_clf.fit(X_train, y_train)

# Predictions
final_blend_predictions = blend_clf.predict(X_test)
final_stacking_predictions = stacking_clf.predict(X_test)
final_adaboost_predictions = ada_pipeline.predict(X_test)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [35]:
from sklearn.metrics import accuracy_score

# ===============================
# 1. Compute Accuracy Scores
# ===============================
log_reg_accuracy = accuracy_score(y_test, log_reg.predict(X_test))
dt_accuracy = accuracy_score(y_test, dt_clf.predict(X_test))
knn_accuracy = accuracy_score(y_test, knn_clf.predict(X_test))

# Make sure ensemble predictions exist (fit them before this step)
# final_blend_predictions = blend_clf.predict(X_test)
# final_stacking_predictions = stacking_clf.predict(X_test)
# final_adaboost_predictions = ada_pipeline.predict(X_test)

blending_accuracy = accuracy_score(y_test, final_blend_predictions)
stacking_accuracy = accuracy_score(y_test, final_stacking_predictions)
adaboost_accuracy = accuracy_score(y_test, final_adaboost_predictions)

# ===============================
# 2. Comparison & Analysis
# ===============================
print("Accuracy Scores:")
print(f"Logistic Regression: {log_reg_accuracy:.4f}")
print(f"Decision Tree: {dt_accuracy:.4f}")
print(f"K-Nearest Neighbors: {knn_accuracy:.4f}")
print(f"Blending: {blending_accuracy:.4f}")
print(f"Stacking: {stacking_accuracy:.4f}")
print(f"AdaBoost: {adaboost_accuracy:.4f}")

print("\nPerformance Comparison and Analysis:")

# Base models
print("\nBase Model Performance:")
print(f"- Logistic Regression: {log_reg_accuracy:.4f} (Strong linear model)")
print(f"- Decision Tree: {dt_accuracy:.4f} (Prone to overfitting, lower accuracy here)")
print(f"- K-Nearest Neighbors: {knn_accuracy:.4f} (Good performance, instance-based)")

# Ensemble vs base
print("\nEnsemble Methods vs. Base Models:")
best_base_accuracy = max(log_reg_accuracy, dt_accuracy, knn_accuracy)
print(f"- Best Base Model Accuracy: {best_base_accuracy:.4f}")

ensemble_methods = {
    "Blending": blending_accuracy,
    "Stacking": stacking_accuracy,
    "AdaBoost": adaboost_accuracy
}

for method, accuracy in ensemble_methods.items():
    if accuracy > best_base_accuracy:
        print(f"- {method} improved performance compared to the best base model.")
    elif accuracy == best_base_accuracy:
        print(f"- {method} achieved the same performance as the best base model.")
    else:
        print(f"- {method} did not improve performance compared to the best base model.")

# Ensemble comparison
print("\nEnsemble Method Performance Comparison:")
for method1, acc1 in ensemble_methods.items():
    for method2, acc2 in ensemble_methods.items():
        if method1 != method2:
            if acc1 > acc2:
                print(f"- {method1} performed better than {method2} ({acc1:.4f} vs {acc2:.4f}).")
            elif acc1 < acc2:
                print(f"- {method1} performed worse than {method2} ({acc1:.4f} vs {acc2:.4f}).")
            else:
                print(f"- {method1} performed the same as {method2} ({acc1:.4f}).")

# Explanation
print("\nAnalysis of Performance Differences:")
print("- Blending and Stacking combine multiple models, improving robustness.")
print("- AdaBoost focuses on misclassified instances iteratively; works well with weak learners.")
print("- Logistic Regression performs well if classes are linearly separable.")
print("- Decision Tree may overfit, explaining lower performance.")
print("- KNN relies on local similarity; performs well if test data resembles training.")

print("\nSummary:")
print("- Logistic Regression was strong as a base model.")
print("- Blending, Stacking, and AdaBoost offered competitive performance.")
print("- Decision Tree showed relatively lower accuracy.")
print("- Ensemble methods can boost robustness, but if base models are already strong, the improvement may be small.")


Accuracy Scores:
Logistic Regression: 0.9182
Decision Tree: 0.9182
K-Nearest Neighbors: 0.9127
Blending: 0.9182
Stacking: 0.9182
AdaBoost: 0.9184

Performance Comparison and Analysis:

Base Model Performance:
- Logistic Regression: 0.9182 (Strong linear model)
- Decision Tree: 0.9182 (Prone to overfitting, lower accuracy here)
- K-Nearest Neighbors: 0.9127 (Good performance, instance-based)

Ensemble Methods vs. Base Models:
- Best Base Model Accuracy: 0.9182
- Blending achieved the same performance as the best base model.
- Stacking did not improve performance compared to the best base model.
- AdaBoost improved performance compared to the best base model.

Ensemble Method Performance Comparison:
- Blending performed better than Stacking (0.9182 vs 0.9182).
- Blending performed worse than AdaBoost (0.9182 vs 0.9184).
- Stacking performed worse than Blending (0.9182 vs 0.9182).
- Stacking performed worse than AdaBoost (0.9182 vs 0.9184).
- AdaBoost performed better than Blending (0.918